# 10 · `gl_engine/interp/program.py`

## What this file is for

ISO's rules arrive as hundreds of XML files per package. This indexes them, so a rule can be called by name — and it holds the answer to a question every earlier analysis in this project got wrong: **where does the program actually start?**

Every census walked `Rule` elements and concluded the entry point was a particular rule. It isn't. Each package carries a `Default` block that is a child of the document root rather than of any rule, and it runs first. An interpreter starting at the rule would have returned a complete, plausible premium with no expiry date and no total.

**Depends on:** [`04-erc-discovery`](04-erc-discovery.ipynb).

## Its public surface

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import inspect
from gl_engine.interp import program

for name, obj in vars(program).items():
    if name.startswith("_") or getattr(obj, "__module__", None) != program.__name__:
        continue
    if inspect.isclass(obj):
        print(f"class {name}")
        for m, f in vars(obj).items():
            if m.startswith("_"):
                continue
            if isinstance(f, property):
                print(f"    .{m}  (property)")
            elif callable(f):
                print(f"    .{m}{inspect.signature(f)}")
    elif inspect.isfunction(obj):
        print(f"def {name}{inspect.signature(obj)}")
    elif name.isupper():
        print(f"{name} = {obj!r}")

## The smallest thing that works

Index one package's rules.

In [ ]:
from gl_engine import EditionResolver
from gl_engine.resolve.book import ResolvedBook
from gl_engine.interp.program import Program

book = ResolvedBook(EditionResolver().resolve("GA", "20260811"))

state = Program(book.state.package)
cw    = Program(book.parent.package)

print(f"{state.pkg_id:<22} {len(state.file_names()):>4} rule files")
print(f"{cw.pkg_id:<22} {len(cw.file_names()):>4} rule files")

The shape of the whole system in two numbers: the countrywide package holds the program, and the state package is a comparatively small set of exceptions layered over it.

## The interesting case

### The entry point is not a rule

In [ ]:
entry = cw.entry()
print("entry element:", entry.tag)
print()
print("It is a `Default` block, a child of the document root -- not a `Rule`.")
print("Anything that enumerated rules could never see it.")

What runs inside it before the main rule matters: the renewal flag defaults, the state code and name are seeded, and **the expiry date is computed as effective date plus one year** — nothing else in the corpus computes it. Start one level too low and every one of those is silently missing.

### Looking up a rule by name

In [ ]:
names = cw.file_names()
print("first few rule files:")
for n in names[:5]:
    print("  ", n)

target = "GeneralLiabilityRules"
print(f"\nhas_file({target!r}):", cw.has_file(target))
rf = cw.file(target)
print("loaded:", type(rf).__name__)

## What it refuses

Calling a rule that doesn't exist fails rather than doing nothing — a silently skipped rule is a missing factor.

In [ ]:
from gl_engine.errors import EngineError

try:
    cw.file("NoSuchRuleFile")
    print("no error")
except (EngineError, KeyError, Exception) as e:
    print(f"{type(e).__name__}: {str(e)[:120]}")

## Try it yourself

1. How many rule files does the state package share a name with in countrywide? Those are the overrides.
2. Open a rule file's XML directly. How many `Rule` elements does one file hold?
3. Find the `Default` block in the state package. Does every package have one?

In [ ]:
# your turn